[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ShozenD/cntmosaic/blob/main/tutorials/03_Tutorial_Prem_Context.ipynb)

# Estimating Context-Specific Contact Matrices with Prem's Model

Social contact patterns differ sharply by where the contact happens. Contacts at home
are dominated by partners and the children; contacts at school are almost
entirely between children of the same age; contacts at work are more diverse. Many
epidemiological questions (school closures, workplace distancing, shielding the elderly) act on a single setting at a time, so modellers need contact matrices split by context.

This tutorial estimates household, school, work, and "other" contact matrices for the United
States using the `Prem2` model, `cntmosaic`'s reformulation of the approach in
[Prem, Cook & Jit (2017)](https://journals.plos.org/ploscompbiol/article?id=10.1371/journal.pcbi.1005697). The data come from the [Epistorm-Mix study](https://www.epistorm.org/data/epistorm-mix), a high-quality post-COVID contact survey in the United States
([Litvinova et al., 2025](https://www.medrxiv.org/content/10.1101/2025.11.20.25340662v1)).

**Technical note.** `Prem2` is a reformulation of Prem et al.'s original estimator: it uses an intrinsic Gaussian Markov random field with a penalised-complexity prior on its precision. [Section 5](#5-how-prem2-estimates-a-contact-matrix) explains what it does.

**Workflow at a glance:**
1. Download and inspect the Epistorm-Mix survey data
2. Preprocess: keep the needed columns and bin ages into 5-year groups
3. Understand what `Prem2` estimates
4. Fit one `Prem2` model per contact setting (household, school, work, other) with MCMC
5. Check MCMC convergence
6. Compare the four context-specific matrices

## Contents
1. [Background: setting-specific contact matrices](#1-background-setting-specific-contact-matrices)
2. [Setup](#2-setup)
3. [The Epistorm-Mix data](#3-the-epistorm-mix-data)
4. [Preprocessing](#4-preprocessing)
5. [How `Prem2` estimates a contact matrix](#5-how-prem2-estimates-a-contact-matrix)
6. [Fitting the four context-specific matrices](#6-fitting-the-four-context-specific-matrices)
7. [Comparing contexts](#7-comparing-contexts)
8. [Summary and next steps](#8-summary-and-next-steps)

## 1. Background: setting-specific contact matrices

A person's total contact pattern is the sum of the contacts they make in different
**settings**: at home, at school, at work, and everywhere else. Writing $M^{\text{total}}$
for the overall age-by-age contact matrix, the decomposition introduced by
[Prem, Cook & Jit (2017)](https://journals.plos.org/ploscompbiol/article?id=10.1371/journal.pcbi.1005697)
is $M^{\text{total}} \approx M^{\text{home}} + M^{\text{school}} + M^{\text{work}} + M^{\text{other}}$.
This is what lets transmission models represent interventions that target one setting -
closing schools zeroes out $M^{\text{school}}$, workplace distancing scales down
$M^{\text{work}}$, and so on.

**Setting is a property of the contact, not the participant.** A survey respondent is not
"in" a setting - instead, each contact they report is tagged with where it took place
(`cont_setting` in Epistorm-Mix). So setting cannot be handled as a participant
stratification variable the way sex or region can. The standard approach, which we follow
here, is:

- take the subset of contacts that occurred in a given setting,
- fit a contact matrix to that subset,
- always divide by the **same** full set of survey participants.

Because every setting-specific matrix uses the same participant denominator $N_c$ (the
number of respondents in age group $c$), the four matrices are - up to the smoothing each
fit applies independently - additive, and each cell reads as "mean number of contacts of
that type, per respondent."

> **Note:** `Prem2` fits each setting independently and does **not** enforce reciprocity
> (the constraint that total contacts from group $c$ to group $d$ equal those from $d$ to
> $c$). Reciprocity adjustment needs population age counts, which Epistorm-Mix does not
> ship; see [Summary and next steps](#8-summary-and-next-steps).

## 2. Setup

### 2.1 Running on Google Colab

If you opened this notebook in Colab, run the cell below once to clone the repository and
install cntmosaic. If you are running locally from a cloned repo it does nothing - skip it.

In [ ]:
# Google Colab only - clones the repo, installs cntmosaic, and sets the working
# directory so the relative data paths below resolve. No-op when run locally.
import os
import subprocess
import sys

try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Detected Google Colab - cloning repository and installing cntmosaic...")
    subprocess.check_call(
        ["git", "clone", "--quiet", "https://github.com/ShozenD/cntmosaic.git"]
    )
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--quiet", "./cntmosaic"]
    )
    os.chdir("cntmosaic/tutorials")
    print(f"Done. Working directory set to: {os.getcwd()}")

### 2.2 Imports and JAX configuration

Two configuration lines matter for the MCMC we run later:

- `jax.config.update("jax_enable_x64", True)` enables 64-bit floating point, which the NUTS
  sampler needs for numerical stability.
- `numpyro.set_host_device_count(4)` lets the 4 MCMC chains run in parallel on CPU.

> **Note on parallel chains:** without `set_host_device_count`, JAX sees a single CPU device
> and NumPyro prints `There are not enough devices to run parallel chains: expected 4 but
> got 1` and runs the chains one after another. The line must come before the first JAX
> operation. On a GPU runtime you can leave it out.

In [ ]:
from pathlib import Path

import pandas as pd
import requests  # only used to download the Epistorm-Mix CSVs

import jax
from jax.random import PRNGKey

jax.config.update("jax_enable_x64", True)

import numpyro

numpyro.set_host_device_count(4)  # run the 4 MCMC chains in parallel

# cntmosaic: data containers, the model, its summariser, the mosaic plot, age bins
from cntmosaic.dataloader import ParticipantData, ContactData
from cntmosaic.models import Prem2
from cntmosaic.analysis import ModelSummariserPrem2
from cntmosaic.vis import plot_mosaic_pixilated
from cntmosaic.utils import AgeGroupSpecs

import altair as alt

## 3. The Epistorm-Mix data

[Epistorm-Mix](https://www.epistorm.org/data/epistorm-mix) is a social contact survey of
roughly 1,930 respondents in the United States, collected in 2024 to measure post-pandemic
mixing patterns
([Litvinova et al., 2025](https://www.medrxiv.org/content/10.1101/2025.11.20.25340662v1)).
We use the project's imputed data files, which fill in missing contact ages:

| File                    | One row per       | Key columns                           |
|-------------------------|-------------------|---------------------------------------|
| `imputed-resp2.csv`     | survey respondent | `resp_id`, `resp_age`, `weight_ipsos` |
| `imputed-contacts2.csv` | reported contact  | `resp_id`, `cont_age`, `cont_setting` |

Each contact links back to its respondent through `resp_id` and carries the setting it
occurred in (`cont_setting`): `Household`, `School`, `Work`, or `Other`.

> **Note:** the survey provides post-stratification weights (`weight_ipsos`). For simplicity
> this tutorial fits an unweighted model; `Prem2` does not apply participant weights.

The cell below downloads the two files into `../data/raw/EpistormMix/` (skipping the
download if they are already present).

In [ ]:
RAW_DATA_PATH = Path("../data/raw/EpistormMix")
RAW_DATA_PATH.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_PATH = Path("../data/processed")

URLS = {
    "imputed-resp2.csv": "https://raw.githubusercontent.com/epistorm/Epistorm-Mix/refs/heads/main/data/imputed_data/imputed-resp2.csv",
    "imputed-contacts2.csv": "https://raw.githubusercontent.com/epistorm/Epistorm-Mix/refs/heads/main/data/imputed_data/imputed-contacts2.csv",
}

for filename, url in URLS.items():
    dest = RAW_DATA_PATH / filename
    if not dest.exists():
        print(f"Downloading {filename}...")
        response = requests.get(url)
        response.raise_for_status()
        dest.write_bytes(response.content)
        print(f"  Saved to {dest}")
    else:
        print(f"Found {filename} locally, skipping download.")

df_resp = pd.read_csv(RAW_DATA_PATH / "imputed-resp2.csv")
df_cnt = pd.read_csv(RAW_DATA_PATH / "imputed-contacts2.csv")

print(f"\nRespondents: {df_resp.shape[0]:,} rows x {df_resp.shape[1]} cols")
print(f"Contacts:    {df_cnt.shape[0]:,} rows x {df_cnt.shape[1]} cols")

Preview the respondent table - one row per person, with their age and survey weight:

In [ ]:
df_resp.head()

And the contact table - one row per reported contact, with the contact's age and the
setting it occurred in:

In [ ]:
df_cnt.head()

### Contacts by setting

How many contacts were reported in each setting? This is the raw material for the four
models - work and household contacts are the most numerous, school the fewest. (These are
raw counts; a few contacts are dropped later by the age filter.)

In [ ]:
df_cnt["cont_setting"].value_counts()

## 4. Preprocessing

`Prem2` needs two tidy tables:

- **participants** - one row per respondent, with an age group;
- **contacts** - one row per contact, with an age group and (for us) the setting.

### 4.1 Keep the columns we need

We drop everything except the identifier and age columns and cast ages to a nullable
integer type. Following cntmosaic's naming convention, participant columns are prefixed
`part_` and contact columns `cnt_`.

In [ ]:
# Participants: keep id + age; contacts: keep id + age + setting
df_resp_slim = df_resp[["resp_id", "resp_age"]].copy()
df_cnt_slim = df_cnt[["resp_id", "cont_age", "cont_setting"]].copy()

# Cast ages to a nullable integer type
df_resp_slim["part_age"] = df_resp_slim["resp_age"].astype("Int64")
df_cnt_slim["cnt_age"] = df_cnt_slim["cont_age"].astype("Int64")

### 4.2 Bin ages into five-year groups

`AgeGroupSpecs(min=0, max=84, step=5)` defines seventeen 5-year bins - `[0, 5)`, `[5, 10)`,
..., `[80, 85)` - where `min` and `max` are *inclusive* age bounds. Its `.cut()` method
turns a column of integer ages into the matching interval labels; both the model and
`plot_mosaic_pixilated` read these bins.

We then restrict to ages 0-84. A few records fall outside this range or have a missing age;
dropping them keeps every age bin populated on both sides, which `Prem2` requires (it raises
an error if any participant age group is empty).

In [ ]:
# Five-year age groups: [0, 5), [5, 10), ..., [80, 85)
age_grp_specs = AgeGroupSpecs(min=0, max=84, step=5)
df_resp_slim["part_age_grp"] = age_grp_specs.cut(df_resp_slim["part_age"])
df_cnt_slim["cnt_age_grp"] = age_grp_specs.cut(df_cnt_slim["cnt_age"])

# Keep ages 0-84 only (drops out-of-range / missing ages so no age bin is empty)
df_resp_slim = df_resp_slim[df_resp_slim["part_age"].between(0, 84)].copy()
df_cnt_slim = df_cnt_slim[df_cnt_slim["cnt_age"].between(0, 84)].copy()

## 5. How `Prem2` estimates a contact matrix

For a single setting, the data reduce to two arrays over the 17 age groups:

- $Y_{cd}$ - the number of contacts that respondents in age group $c$ reported with people
  in age group $d$;
- $N_c$ - the number of respondents in age group $c$.

The quantity we want is the **contact intensity** $M_{cd}$: the mean number of contacts (in
this setting) that a respondent aged $c$ has with people aged $d$. The obvious estimate
$Y_{cd}/N_c$ is very noisy wherever a cell has few respondents or few contacts - exactly
where survey data are thinnest (young-old cells, the oldest ages).

`Prem2` fits a smooth surface through the log-intensity instead. With the default
negative-binomial likelihood the model is $Y_{cd} \sim \text{NegBin}(\mu_{cd},\,\phi)$
with $\mu_{cd} = N_c \exp(\beta_0 + f_{cd})$, so the contact intensity is
$M_{cd} = \exp(\beta_0 + f_{cd})$. Here $\beta_0$ is a single baseline level and $f$ is a
17x17 surface of age-by-age deviations from it (only their sum $\beta_0 + f_{cd}$ is
identified by the data - [Section 6.5](#65-checking-convergence) returns to this). The $N_c$
term enters as a fixed offset, so $Y$ is modelled as counts while $M$ is a per-respondent
rate.

**The smoothing prior.** $f$ is given a first-order 2-D *intrinsic Gaussian Markov random
field* (IGMRF) prior. Informally, this penalises the squared difference between neighbouring
cells of $f$ in both the participant-age and contact-age directions - a 2-D random walk - so
each cell borrows strength from its neighbours and the estimated surface is smooth. A single
precision parameter $\tau$ controls how smooth: `Prem2` places a *penalised-complexity*
prior on $\tau$ (equivalent to $\tau^{-1/2} \sim \text{Exponential}(2.86)$ on the marginal
standard deviation), which shrinks towards the smooth solution unless the data clearly
support more structure.

|             | Prem et al. (2017)             | `Prem2` (this notebook)                |
|-------------|--------------------------------|----------------------------------------|
| Inference   | plug-in estimate + bootstrap   | full Bayesian (NUTS or SVI)            |
| Smoothing   | applied post hoc               | IGMRF prior inside the model           |
| Likelihood  | -                              | negative binomial (default) or Poisson |
| Uncertainty | bootstrap resampling           | posterior credible intervals           |
| Reciprocity | enforced using population data | not enforced (no population data)      |

> **Note:** the original `Prem` model in cntmosaic fits the same idea to individual contact
> records and can add participant-level random effects. `Prem2` works with aggregated count
> cells, is faster, and is the recommended starting point.

## 6. Fitting the four context-specific matrices

We now fit one `Prem2` model per setting. The pattern is the same each time:

1. wrap the respondents in a `ParticipantData` container (built **once**, reused for every
   setting) and the setting's contacts in a `ContactData` container;
2. construct `Prem2` and run MCMC;
3. wrap the result in `ModelSummariserPrem2` to get posterior summaries.

`ParticipantData` and `ContactData` validate their inputs (checking column types, dropping
rows with missing ages) and standardise column names for the model.

### 6.1 Household

`cnt_hh_data` is just the `cont_setting == "Household"` slice of the contacts; `part_data`
stays the full respondent set.

In [ ]:
part_data = ParticipantData(
    data=df_resp_slim, id_col="resp_id", age_grp_col="part_age_grp"
)

# Household contacts
cnt_hh_data = ContactData(
    data=df_cnt_slim[df_cnt_slim["cont_setting"] == "Household"],
    id_col="resp_id",
    age_grp_col="cnt_age_grp",
)

`run_inference_mcmc` runs the No-U-Turn Sampler (NUTS). The main arguments:

| Argument             | Meaning                           | Value here    |
|----------------------|-----------------------------------|---------------|
| `prng_key`           | JAX random seed (reproducibility) | `PRNGKey(1)`  |
| `num_warmup`         | adaptation steps, discarded       | 500           |
| `num_samples`        | posterior draws kept per chain    | 1000          |
| `num_chains`         | independent chains                | 4             |
| `target_accept_prob` | NUTS target acceptance rate       | 0.8 (default) |
| `max_tree_depth`     | maximum NUTS trajectory length    | 10 (default)  |

After sampling, the model prints `Number of divergences: 0` (and again as
`Number of divergent transitions: 0`). Zero is what we want; a non-zero count means the
sampler struggled with the posterior geometry and the estimates may be biased - we hit
exactly this for the school model in
[Section 6.2](#62-school-responding-to-divergences).

Constructing `ModelSummariserPrem2(prem_hh)` prints a warning:
`PopulationData not provided. Reciprocity adjustment and depixilation will not be possible.`
**This is expected.** Epistorm-Mix has no population age file, so contact *rates*,
reciprocity adjustment, and 1-year-resolution matrices are unavailable. The
contact-*intensity* matrix we plot here needs none of them.

> **Note:** MCMC gives exact posterior samples but is the slow option. For quick
> exploration, `prem_hh.run_inference_svi(PRNGKey(0), num_steps=20_000)` runs variational
> inference instead; everything downstream (`ModelSummariserPrem2`, the plots) is
> identical.

In [ ]:
# Initialize the Prem2 model
prem_hh = Prem2(
    part_data=part_data, cnt_data=cnt_hh_data, age_group_specs=age_grp_specs
)

# Run MCMC inference
prng_key = PRNGKey(1)
prem_hh.run_inference_mcmc(prng_key, num_warmup=500, num_samples=1000, num_chains=4)

# Summarize the results
summariser_hh = ModelSummariserPrem2(prem_hh)

`summarise_cint()` returns a dictionary of posterior summaries keyed by `"source->target"`
stratum. This model is unstratified, so there is a single key, `"All->All"`. Each value is a
`ContactSummary` with `.central` (the posterior median matrix), `.lower`, and `.upper` (the
95% credible interval), and it can be passed straight to `plot_mosaic_pixilated`.

`plot_mosaic_pixilated` draws the matrix at age-group ("pixel") resolution: each block is
one age-group pair, block size shows the bin width, colour shows the intensity, and the
y-axis runs bottom-to-top from youngest to oldest.

In [ ]:
summ_cint_hh = summariser_hh.summarise_cint()
plot_mosaic_pixilated(
    summ_cint_hh["All->All"],
    age_grp_specs,
    zlabel="Intensity",
    title="Household Contacts",
    color_min=0.0,
    color_max=0.7,
)

**Interpreting the results:**
- **Strong leading diagonal**: assortative mixing: partners and similarly-aged household
  members.
- **Two off-diagonal bands ~25-35 years off the diagonal**: the parent-child relationship,
  seen from both sides (a 35-year-old with a 5-year-old, and vice versa).

### 6.2 School: responding to divergences

School contacts are concentrated in a handful of child age cells, which gives the posterior
a sharp, awkward geometry. With the default settings, NUTS reports **divergent transitions**
for this model and warns
`Found N divergent transitions. Consider increasing target_accept_prob or max_tree_depth.`

The fix, already applied in the cell below, is to make the sampler take smaller, more
careful steps: `target_accept_prob=0.98` (from 0.8) and `max_tree_depth=13` (from 10). This
removes the divergences at the cost of a slower run - the first chain takes ~20 seconds.
Raising these two settings is the standard first response to divergences.

In [ ]:
cnt_sc_data = ContactData(
    data=df_cnt_slim[df_cnt_slim["cont_setting"] == "School"],
    id_col="resp_id",
    age_grp_col="cnt_age_grp",
)

prem_sc = Prem2(
    part_data=part_data, cnt_data=cnt_sc_data, age_group_specs=age_grp_specs
)

# Tighter NUTS settings than the default to remove divergent transitions:
# smaller steps (target_accept_prob) and longer trajectories (max_tree_depth).
prng_key = PRNGKey(1)
prem_sc.run_inference_mcmc(
    prng_key,
    num_warmup=500,
    num_samples=1000,
    num_chains=4,
    target_accept_prob=0.98,
    max_tree_depth=13,
)

summariser_sc = ModelSummariserPrem2(prem_sc)

In [ ]:
summ_cint_sc = summariser_sc.summarise_cint()
plot_mosaic_pixilated(
    summ_cint_sc["All->All"],
    age_grp_specs,
    zlabel="Intensity",
    title="School Contacts",
    color_min=0.0,
    color_max=2.8,
)

**Interpreting the results:**
- **Dominant diagonal at young ages**: classmates of the same age.
- **Faint band at ~30-60**: teachers and school staff.
- **Near-zero everywhere else**: no school contacts outside education ages.

### 6.3 Work

Work contacts fill in the adult ages. No sampler tweaks are needed here - the defaults give
zero divergences.

In [ ]:
cnt_wk_data = ContactData(
    data=df_cnt_slim[df_cnt_slim["cont_setting"] == "Work"],
    id_col="resp_id",
    age_grp_col="cnt_age_grp",
)

prem_wk = Prem2(
    part_data=part_data, cnt_data=cnt_wk_data, age_group_specs=age_grp_specs
)

prng_key = PRNGKey(1)
prem_wk.run_inference_mcmc(prng_key, num_warmup=500, num_samples=1000, num_chains=4)

summariser_wk = ModelSummariserPrem2(prem_wk)

In [ ]:
summ_cint_wk = summariser_wk.summarise_cint()
plot_mosaic_pixilated(
    summ_cint_wk["All->All"],
    age_grp_specs,
    zlabel="Intensity",
    title="Work Contacts",
    color_min=0,
    color_max=0.7,
)

**Interpreting the results:**
- **Broad working-age block (~20-65)** - colleagues span a wide age range, so the pattern is
  much less diagonal than school.
- **Empty rows and columns for children and the retired** - as expected.

### 6.4 Other

"Other" collects every contact that was not at home, school, or work - shops, transport,
restaurants, leisure, and so on.

In [ ]:
cnt_ot_data = ContactData(
    data=df_cnt_slim[df_cnt_slim["cont_setting"] == "Other"],
    id_col="resp_id",
    age_grp_col="cnt_age_grp",
)

prem_ot = Prem2(
    part_data=part_data, cnt_data=cnt_ot_data, age_group_specs=age_grp_specs
)

prng_key = PRNGKey(1)
prem_ot.run_inference_mcmc(prng_key, num_warmup=500, num_samples=1000, num_chains=4)

summariser_ot = ModelSummariserPrem2(prem_ot)

In [ ]:
summ_cint_ot = summariser_ot.summarise_cint()
plot_mosaic_pixilated(
    summ_cint_ot["All->All"],
    age_grp_specs,
    zlabel="Intensity",
    title="Other Contacts",
    color_min=0.0,
    color_max=0.7,
)

**Interpreting the results:**
- **Weak diagonal, broad off-diagonal support** - casual contacts in public spaces are less
  age-assortative than household or school contacts.
- **Some parent-child structure** - families out together.

### 6.5 Checking convergence

Before trusting any of these matrices, check that the four chains actually converged.
`prem_hh._mcmc_result` is the underlying NumPyro `MCMC` object, which
[ArviZ](https://python.arviz.org) can read directly (the feature-selection tutorial,
`04_Tutorial_Feature_Selection.ipynb`, surfaces the same object as `result.idatas`).

We monitor `log_cint` - the log contact-intensity surface $\beta_0 + f$ that the matrix is
built from - together with the scalars `tau` (smoothing precision) and `inv_disp`
(dispersion). We deliberately do **not** check `baseline` and `f` separately: the IGMRF
prior does not pin the overall level of `f`, and the likelihood only ever sees the sum
$\beta_0 + f_{cd}$, so those two are not individually identified. Only `log_cint` is, and
that is all the matrix depends on. `az.summary(idata_hh, var_names=["baseline"])` on its own
will look unconverged; that is expected and harmless here.

In [ ]:
import arviz as az

idata_hh = az.from_numpyro(prem_hh._mcmc_result)

# tau is the smoothing precision; inv_disp is the negative-binomial dispersion.
# Want r_hat close to 1.00 and ess in the hundreds or more.
az.summary(idata_hh, var_names=["tau", "inv_disp"])

Both should have `r_hat` close to 1.00 (below 1.01) and `ess_bulk` / `ess_tail` (effective
sample sizes) of at least a few hundred - enough for stable medians and 95% intervals.

`log_cint` has 289 cells, so summarise its diagnostics rather than printing every row:

In [ ]:
# log_cint = baseline + f is the log contact-intensity surface (289 cells) - the
# quantity the matrix is actually built from. Summarise its diagnostics rather
# than printing every row.
az.summary(idata_hh, var_names=["log_cint"])[["r_hat", "ess_bulk"]].describe()

> **Note:** run the same check on `prem_sc`, `prem_wk`, and `prem_ot`. The school model is
> the one to watch, since it needed the tighter sampler settings.

## 7. Comparing contexts

Finally, place the four posterior-median matrices side by side.

**A note on colour scales.** The cell below gives each panel its **own** colour scale
(`resolve_scale(color="independent")`). This makes the structure within each setting easy to
see, but it hides the large differences in magnitude between settings - the school maximum
is several times the others. To compare magnitudes directly, set a single shared
`color_min` / `color_max` across all four panels and drop `resolve_scale`.

In [ ]:
chart_hh = plot_mosaic_pixilated(
    summ_cint_hh["All->All"],
    age_grp_specs,
    zlabel="Intensity",
    xlabel="Age of respondent",
    ylabel="Age of contact",
    title="Household",
    width=200,
    height=200,
    color_min=0.0,
    color_max=0.7,
)

chart_sc = plot_mosaic_pixilated(
    summ_cint_sc["All->All"],
    age_grp_specs,
    zlabel="Intensity",
    xlabel="Age of respondent",
    title="School",
    show_y_label=False,
    show_y_ticks=False,
    width=200,
    height=200,
    color_min=0.0,
    color_max=2.8,
)

chart_wk = plot_mosaic_pixilated(
    summ_cint_wk["All->All"],
    age_grp_specs,
    zlabel="Intensity",
    xlabel="Age of respondent",
    ylabel="Age of contact",
    title="Work",
    show_y_label=True,
    show_y_ticks=True,
    width=200,
    height=200,
    color_min=0.0,
    color_max=0.7,
)

chart_ot = plot_mosaic_pixilated(
    summ_cint_ot["All->All"],
    age_grp_specs,
    zlabel="Intensity",
    xlabel="Age of respondent",
    title="Other",
    show_y_label=False,
    show_y_ticks=False,
    width=200,
    height=200,
    color_min=0.0,
    color_max=0.7,
)

top_row = alt.hconcat(chart_hh, chart_sc)
bottom_row = alt.hconcat(chart_wk, chart_ot)
final_chart = alt.vconcat(top_row, bottom_row).resolve_scale(color="independent")

display(final_chart)

**Interpreting the results:**
- **Household** - tight diagonal plus parent-child bands; low overall intensity.
- **School** - a spike on the child diagonal, near-zero elsewhere, and by far the highest
  intensities.
- **Work** - a broad adult block, weakly diagonal.
- **Other** - diffuse and weakly age-assortative.

Each cell is a mean number of contacts per respondent in that setting, so - up to the
smoothing each model applied independently - the four matrices sum to the all-contexts
contact matrix.

## 8. Summary and next steps

In this tutorial you:
- downloaded and inspected the Epistorm-Mix US contact survey;
- binned ages into 5-year groups and split contacts by setting;
- fitted one `Prem2` model per setting (household, school, work, other) with MCMC,
  tightening the sampler for the school model to remove divergences;
- checked convergence with ArviZ (`r_hat`, effective sample size);
- compared the four context-specific contact matrices.

**Where to go next:**
- **Add population data** - pass a `PopulationData` to `ModelSummariserPrem2` (for the US,
  `cntmosaic.datasets.load_age_distribution("United_States", 84)`) to unlock reciprocity
  adjustment (`apply_reciprocity=True`), per-capita contact **rates** (`summarise_rate`),
  and 1-year-resolution matrices (`summarise_cint(return_depixilated=True)`).
- **Stratify further** - add respondent sex or region via `strat_var_cols` on
  `ParticipantData`; see `01_Tutorial_GenMix.ipynb`.
- **Faster inference** - swap `run_inference_mcmc` for `run_inference_svi`.
- **Other smoothing priors** - `PSpline2D`, `HSGP2D`, `vdKassteele2D` in place of the IGMRF.
- **API reference** - `Prem2`, `ModelSummariserPrem2`, `plot_mosaic_pixilated`.